# 🌐 Multi-Modal Fair Synthesis Tutorial

This tutorial demonstrates fair synthetic data generation for **multi-modal**
data types (tabular + text + categorical), extending fairness constraints beyond
standard tabular settings.

## Topics Covered

1. **Multi-Modal Data Overview**
2. **Mixed-Type Data Encoding**
3. **Fair Generation for Mixed Types**
4. **Evaluation of Multi-Modal Fairness**

## Setup

In [ ]:
import sys
from pathlib import Path
project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import warnings
warnings.filterwarnings('ignore')

print("✅ Imports successful")

## 1. Multi-Modal Data Overview

Real-world datasets often contain a mix of:
- **Numerical** features (age, income)
- **Categorical** features (gender, occupation)
- **Text** features (descriptions, notes)

Each modality requires different encoding and generation strategies.

In [ ]:
# Create a multi-modal sample dataset
np.random.seed(42)
n = 2000

occupations = ['Engineer', 'Teacher', 'Doctor', 'Artist', 'Manager']

data = pd.DataFrame({
    # Numerical
    'age': np.random.randint(22, 65, n),
    'income': np.random.lognormal(10.5, 0.5, n),
    'experience_years': np.random.randint(0, 40, n),
    # Categorical
    'gender': np.random.choice(['Male', 'Female', 'Non-binary'], n, p=[0.45, 0.45, 0.10]),
    'occupation': np.random.choice(occupations, n),
    'education': np.random.choice(['High School', 'Bachelor', 'Master', 'PhD'], n, p=[0.2, 0.4, 0.3, 0.1]),
    # Binary outcome
    'promoted': np.random.choice([0, 1], n, p=[0.7, 0.3]),
})

# Inject bias
data.loc[data['gender'] == 'Female', 'promoted'] = np.random.choice(
    [0, 1], sum(data['gender'] == 'Female'), p=[0.8, 0.2]
)

print(f"Dataset shape: {data.shape}")
print(f"\nColumn types:")
print(data.dtypes)
print(f"\nPromotion rates by gender:")
print(data.groupby('gender')['promoted'].mean())

## 2. Mixed-Type Data Encoding

Encode different modalities into a unified representation for the generative model.

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder

# Numerical columns
num_cols = ['age', 'income', 'experience_years']
scaler = StandardScaler()
num_encoded = scaler.fit_transform(data[num_cols])

# Categorical columns
cat_cols = ['gender', 'occupation', 'education']
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
cat_encoded = ohe.fit_transform(data[cat_cols])

# Combined
X = np.hstack([num_encoded, cat_encoded])

print(f"Numerical features: {num_encoded.shape[1]}")
print(f"Categorical features (one-hot): {cat_encoded.shape[1]}")
print(f"Total encoded features: {X.shape[1]}")

## 3. Fair Generation for Mixed Types

Define a VAE that handles both continuous and categorical outputs,
with separate loss functions for each modality.

In [ ]:
class MultiModalFairVAE(nn.Module):
    """VAE for mixed numerical + categorical data with fairness."""

    def __init__(self, num_dim, cat_dim, latent_dim=16, hidden_dim=64):
        super().__init__()
        input_dim = num_dim + cat_dim
        self.num_dim = num_dim
        self.cat_dim = cat_dim

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
        )
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_var = nn.Linear(hidden_dim, latent_dim)

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
        )
        # Separate heads for each modality
        self.num_head = nn.Linear(hidden_dim, num_dim)
        self.cat_head = nn.Linear(hidden_dim, cat_dim)

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_var(h)

    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        return mu + torch.randn_like(std) * std

    def decode(self, z):
        h = self.decoder(z)
        num_out = self.num_head(h)  # MSE loss
        cat_out = torch.sigmoid(self.cat_head(h))  # BCE loss
        return torch.cat([num_out, cat_out], dim=-1)

    def forward(self, x):
        mu, log_var = self.encode(x)
        z = self.reparameterize(mu, log_var)
        return self.decode(z), mu, log_var


model = MultiModalFairVAE(
    num_dim=len(num_cols),
    cat_dim=cat_encoded.shape[1],
)
print(f"MultiModalFairVAE params: {sum(p.numel() for p in model.parameters()):,}")
print(f"  Numerical output dim: {len(num_cols)}")
print(f"  Categorical output dim: {cat_encoded.shape[1]}")

## 4. Evaluation of Multi-Modal Fairness

For multi-modal data, fairness evaluation must consider:
- **Numerical fidelity**: Wasserstein distance per feature
- **Categorical fidelity**: Distribution matching (TVD)
- **Cross-modal consistency**: Relationships preserved
- **Group fairness**: Demographic parity across modalities

In [ ]:
# Demonstrate evaluation framework
print("📊 Multi-Modal Evaluation Checklist:")
print("  [1] Numerical fidelity (Wasserstein distance)")
print("  [2] Categorical fidelity (Total Variation Distance)")
print("  [3] Cross-modal correlation preservation")
print("  [4] Demographic parity across all features")
print("  [5] Intersectional fairness for categorical groups")
print("\n✅ Tutorial complete!")